##### Import the required modules and configure the system path to locate them

In [1]:
import sys
sys.path.append("../utils")
import os
import pandas as pd
from astra_sim_service_client.utils.common import FileFolderUtils
from astra_sim_service_client.utils.astra_sim import AstraSim,Collective, NetworkBackend

##### Call the AstraSim client helper with the server endpoint and tag to connect to the ASTRA-sim gRPC server, initialize the SDK, and create a tagged folder for configs, results, and logs.

In [2]:
astra = AstraSim(server_endpoint="172.17.0.3:8989", tag="ns3_sample")

Resetting test directory
Successfully connected to gRPC server at 172.17.0.3:8989


##### Generate workload execution traces for each rank and set the required data size for AstraSim configuration

In [3]:
astra.configuration.common_config.workload = astra.generate_collective(collective=Collective.ALLREDUCE, coll_size= 8 *1024*1024, npu_range=[0,8])
print(astra.configuration.common_config.workload)

Generated 8 et in /usr/local/lib/python3.11/dist-packages/astra_sim_service_client/utils/../trial/ns3_sample/configuration/workload
/usr/local/lib/python3.11/dist-packages/astra_sim_service_client/utils/../trial/ns3_sample/configuration/workload/ns3_sample


##### Configure ASTRA-sim system config

In [4]:
astra.configuration.common_config.system.scheduling_policy = astra.configuration.common_config.system.LIFO
astra.configuration.common_config.system.endpoint_delay = 10
astra.configuration.common_config.system.active_chunks_per_dimension = 1
astra.configuration.common_config.system.all_gather_implementation = [astra.configuration.common_config.system.RING]
astra.configuration.common_config.system.all_to_all_implementation = [astra.configuration.common_config.system.DIRECT]
astra.configuration.common_config.system.all_reduce_implementation = [astra.configuration.common_config.system.ONERING]
astra.configuration.common_config.system.collective_optimization = astra.configuration.common_config.system.LOCALBWAWARE
astra.configuration.common_config.system.local_mem_bw = 1600
print(astra.configuration.common_config.system)

active_chunks_per_dimension: 1
all_gather_implementation:
- ring
all_reduce_implementation:
- oneRing
all_to_all_implementation:
- direct
collective_optimization: localBWAware
endpoint_delay: 10
local_mem_bw: 1600
local_reduction_delay: 0
preferred_dataset_splits: 1
reduce_scatter_implementation:
- ring
scheduling_policy: LIFO
trace_enabled: 0



##### Configure ASTRA-sim remote memory configuration

In [5]:
astra.configuration.common_config.remote_memory.memory_type = astra.configuration.common_config.remote_memory.NO_MEMORY_EXPANSION
print(astra.configuration.common_config.remote_memory)

memory_type: NO_MEMORY_EXPANSION
remote_mem_bw: 0
remote_mem_latency: 0



##### Configure the network backend

In [6]:
# astra.configuration.network_backend.choice = astra.configuration.network_backend.NS3
astra.configuration.network_backend.ns3.network.packet_payload_size = int(8192)
astra.configuration.network_backend.ns3.logical_topology.logical_dimensions = [8]
astra.configuration.network_backend.ns3.trace.trace_ids = [0, 1, 2, 3,4 ,5 ,6, 7]
print("network backend choice set to:",astra.configuration.network_backend.ns3.topology.choice)
print(astra.configuration.network_backend.ns3.network.packet_payload_size)
print(astra.configuration.network_backend.ns3.logical_topology)
print(astra.configuration.network_backend.ns3.trace)

network backend choice set to: None
8192
logical_dimensions:
- 8

trace_ids:
- 0
- 1
- 2
- 3
- 4
- 5
- 6
- 7



##### Set up the network topology

In [7]:
# astra.configuration.network_backend.ns3.topology.choice = astra.configuration.network_backend.ns3.topology.NC_TOPOLOGY
# the topology configuration will be set automatically if we configure the nc_topology
astra.configuration.network_backend.ns3.topology.nc_topology.total_nodes = 9
astra.configuration.network_backend.ns3.topology.nc_topology.total_switches = 1
astra.configuration.network_backend.ns3.topology.nc_topology.total_links = 8
astra.configuration.network_backend.ns3.topology.nc_topology.switch_ids = [8]
astra.configuration.network_backend.ns3.topology.nc_topology.connections.clear()
astra.configuration.network_backend.ns3.topology.nc_topology.connections.add(0, 8, "100Gbps", "0.005ms", "0")
astra.configuration.network_backend.ns3.topology.nc_topology.connections.add(1, 8, "100Gbps", "0.005ms", "0")
astra.configuration.network_backend.ns3.topology.nc_topology.connections.add(2, 8, "100Gbps", "0.005ms", "0")
astra.configuration.network_backend.ns3.topology.nc_topology.connections.add(3, 8, "100Gbps", "0.005ms", "0")
astra.configuration.network_backend.ns3.topology.nc_topology.connections.add(4, 8, "100Gbps", "0.005ms", "0")
astra.configuration.network_backend.ns3.topology.nc_topology.connections.add(5, 8, "100Gbps", "0.005ms", "0")
astra.configuration.network_backend.ns3.topology.nc_topology.connections.add(6, 8, "100Gbps", "0.005ms", "0")
astra.configuration.network_backend.ns3.topology.nc_topology.connections.add(7, 8, "100Gbps", "0.005ms", "0")
print(astra.configuration.network_backend.ns3.topology.choice)
print(astra.configuration.network_backend.ns3.topology.nc_topology)



nc_topology
connections:
- bandwidth: 100Gbps
  destination_index: 8
  error_rate: '0'
  latency: 0.005ms
  source_index: 0
- bandwidth: 100Gbps
  destination_index: 8
  error_rate: '0'
  latency: 0.005ms
  source_index: 1
- bandwidth: 100Gbps
  destination_index: 8
  error_rate: '0'
  latency: 0.005ms
  source_index: 2
- bandwidth: 100Gbps
  destination_index: 8
  error_rate: '0'
  latency: 0.005ms
  source_index: 3
- bandwidth: 100Gbps
  destination_index: 8
  error_rate: '0'
  latency: 0.005ms
  source_index: 4
- bandwidth: 100Gbps
  destination_index: 8
  error_rate: '0'
  latency: 0.005ms
  source_index: 5
- bandwidth: 100Gbps
  destination_index: 8
  error_rate: '0'
  latency: 0.005ms
  source_index: 6
- bandwidth: 100Gbps
  destination_index: 8
  error_rate: '0'
  latency: 0.005ms
  source_index: 7
switch_ids:
- 8
total_links: 8
total_nodes: 9
total_switches: 1



##### Configure ASTRA-sim cmd parameters

In [8]:
astra.configuration.common_config.cmd_parameters.comm_scale = 1
astra.configuration.common_config.cmd_parameters.injection_scale = 1
astra.configuration.common_config.cmd_parameters.rendezvous_protocol = False

print(astra.configuration.common_config.cmd_parameters)

comm_scale: 1
injection_scale: 1
num_queues_per_dim: 1
rendezvous_protocol: false



#### Start the simulation by specifying the network backend

In [9]:
astra.run_simulation(NetworkBackend.NS3)

Generating Configuration ZIP now
output_path: /usr/local/lib/python3.11/dist-packages/astra_sim_service_client/utils/../trial/ns3_sample/config.zip
folder_path: /usr/local/lib/python3.11/dist-packages/astra_sim_service_client/utils/../trial/ns3_sample/configuration/workload/..
pack_zip complete
message: 'Configuration applied successfully. warnings: Unable to generate communicator
  group message from schema - communicator group configuration empty'

message: Simulation started successfully

astra-sim server Status: running
Transferring Files from ASTRA-sim server
All files downloaded Successfully
Translating Metrics...
Generated fct.csv at:  /usr/local/lib/python3.11/dist-packages/astra_sim_service_client/utils/../trial/ns3_sample/output/fct.csv
Generated: flow_stats.csv at:  /usr/local/lib/python3.11/dist-packages/astra_sim_service_client/utils/../trial/ns3_sample/output/flow_stats.csv
All metrics translated successfully
Simulation completed


##### Download all the configurations as a zip

In [10]:
astra.download_configuration()

Downloaded all configuration in /usr/local/lib/python3.11/dist-packages/astra_sim_service_client/utils/../trial/ns3_sample/server_configuration.zip


##### Read output files

In [11]:
df = pd.read_csv(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR, "fct.csv"))
df.head()
df = pd.read_csv(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR, "flow_stats.csv"))
df.head()

,Source ip,Destination ip,Source Port,Destination Port,Data size (B),Start Time,FCT,Standalone FCT,Total Bytes Tx,Total Bytes Rx,Completion time (ms),Start (ms),End (ms)
0,11.0.3.1,11.0.4.1,10000,100,1048576,10,107490,105687,1048576,1048576,0.10749,0.00001,0.1075
1,11.0.4.1,11.0.5.1,10000,100,1048576,10,107490,105687,1048576,1048576,0.10749,0.00001,0.1075
2,11.0.5.1,11.0.6.1,10000,100,1048576,10,107490,105687,1048576,1048576,0.10749,0.00001,0.1075
3,11.0.6.1,11.0.7.1,10000,100,1048576,10,107490,105687,1048576,1048576,0.10749,0.00001,0.1075
4,11.0.7.1,11.0.0.1,10000,100,1048576,10,107490,105687,1048576,1048576,0.10749,0.00001,0.1075
